In [0]:
from pyspark.sql import functions as f
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/shivamk66297@gmail.com/consolidated_pipeline/1_setup/utilities

In [0]:
print(bronze_schema,silver_schema,gold_schema)

In [0]:
dbutils.widgets.text("catalog","fmcg","Catalog")
dbutils.widgets.text("data_source","customers","Data Source")

In [0]:
catalog=dbutils.widgets.get("catalog")
data_source=dbutils.widgets.get("data_source")
base_path=f's3://sportsbar-atlicon-dp/{data_source}/*.csv'

In [0]:
df=(
    spark.read.format('csv')
              .option('header',True)
              .option('inferSchema',True)
              .load(base_path)
              .withColumn('read_timestamp',f.current_timestamp())
              .select("*","_metadata.file_name","_metadata.file_size")
)
display(df.limit(10))

In [0]:
df.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed","true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")    

In [0]:
df_bronze=spark.sql(f"select * from {catalog}.{bronze_schema}.{data_source}")
display(df_bronze.limit(10))

In [0]:
df_duplicate=df_bronze.groupBy("customer_id").count().where("count>1")
df_duplicate.show()

In [0]:
# Remove Duplicates

print("row before",df_bronze.count())
df_silver=df_bronze.dropDuplicates(["customer_id"])
print("row after",df_silver.count())

In [0]:
# Trim leading space

df_silver=df_silver.withColumn(
    'customer_name',
    f.trim(f.col("customer_name"))
)

In [0]:
df_silver.filter(f.col("customer_name")!=f.trim(f.col("customer_name"))).show()

In [0]:
df_silver.select('city').distinct().show()

In [0]:
# Incorrect city -> correct city

city_mapping = {
    "Bengalore": "Bengaluru",
    "Bengaluruu": "Bengaluru",

    "Hyderabadd": "Hyderabad",
    "Hyderbad": "Hyderabad",

    "NewDelhee": "New Delhi",
    "NewDelhi": "New Delhi",
    "NewDheli": "New Delhi"
}

allowed=["Bengaluru","Hyderabad","New Delhi"]

df_silver=(
    df_silver
    .replace(city_mapping,subset=["city"])
    .withColumn(
        "city",
        f.when(f.col("city").isNull(),None)
         .when(f.col("city").isin(allowed),f.col("city"))
         .otherwise(None)
    )
)
df_silver.select('city').distinct().show()

In [0]:
df_silver.select('customer_name').distinct().show()

In [0]:
# Captialize lower_case into upperCase
# initcap() use to captialize first letter of a words

df_silver=df_silver.withColumn(
    "customer_name",
    f.when(f.col("customer_name").isNull(),None)
    .otherwise(f.initcap("customer_name"))
)
df_silver.select('customer_name').distinct().show()

In [0]:
df_silver.filter(f.col("city").isNull()).show(truncate=True)

In [0]:
null_customer_name=['Sprintx Nutrition','Zenathlete Foods','Primefuel Nutrition','Recovery Lane']
df_silver.filter(f.col("customer_name").isin(null_customer_name)).show(truncate=True)

In [0]:
# Business confirmation note : City correction confirmed by business team

customer_city_fix={
    # Sprintx Nutrition
    789403:"New Delhi",
    # Zenathlete Foods
    789420:"Bengaluru",
    # Primefuel Nutrition
    789521:"Hyderabad",
    #Recovery Lane
    789603:"Hyderabad"


}

df_fix=spark.createDataFrame(
    [(k,v) for k,v in customer_city_fix.items()],
    ['customer_id','fixed_city']
)

In [0]:
df_silver=(
        df_silver
        .join(df_fix,on="customer_id",how="left")
        .withColumn(
            "city",
            f.coalesce("city","fixed_city")
        )
        .drop("fixed_city")
    )
display(df_silver)

In [0]:
df_silver=df_silver.withColumn("customer_id",f.col("customer_id").cast("string"))
print(df_silver.printSchema())

In [0]:
df_silver=(
    df_silver
    # building final customer column:"customerName-city" or "customerName-Unknown"
    .withColumn(
        "customer",
        f.concat_ws("-","customer_name",f.coalesce(f.col("city"),f.lit("Unknow")))
    )
    # Static attribute aligned with parent data model
    .withColumn("market",f.lit("India"))
    .withColumn("platform",f.lit("Sports Bar"))
    .withColumn("channel",f.lit("Acquisition"))
)

display(df_silver)

In [0]:
df_silver.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed","true")\
    .option("mergeSchema","true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

## Gold processing

In [0]:
df_gold=df_silver.select("customer_id","customer_name","city","customer","market","platform","channel")

In [0]:
df_gold.write\
        .format('delta')\
        .option('delta.enableChangeDataFeed','true')\
        .mode("overwrite")\
        .saveAsTable(f"{catalog}.{gold_schema}.sb_dim_{data_source}")

In [0]:
delta_table=DeltaTable.forName(spark,"fmcg.gold.dim_customers")
df_child_customers=spark.table("fmcg.gold.sb_dim_customers").select(
    f.col("customer_id").alias("customer_code"),
    "customer",
    "market",
    "platform",
    "channel"
)

In [0]:
delta_table.alias("target").merge(
    source=df_child_customers.alias('source'),
    condition="target.customer_code = source.customer_code"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()